# Solutions generation

Generating solutions of ill-defined problems using different LLM-based baselines.

In [ ]:
MODEL = 'deepseek/deepseek-v4-flash'
TEMPERATURE = 0.5

In [ ]:
from langchain_openai import ChatOpenAI
from fp2mp_baselines.config import config

llm = ChatOpenAI(
    model=MODEL,
    base_url=config.base_url,
    api_key=config.api_key,
    temperature=TEMPERATURE,
)

In [8]:
from src import read_json, write_json

problems = read_json('./data/problems.json')

In [9]:
from fp2mp_baselines.single_agent import build_single_agent_graph
from fp2mp_baselines.cot import build_cot_graph
from fp2mp_baselines.generator_critic import build_generator_critic_graph

baselines = {
    'single-agent' : build_single_agent_graph(llm),
    'chain-of-thoughts': build_cot_graph(llm),
    'generator-critic': build_generator_critic_graph(llm) 
}

In [10]:
from tqdm import tqdm

solutions = []

for bs_name, bs_graph in baselines.items():
    for problem in tqdm(problems):
        state = bs_graph.invoke({'input': problem})
        solution = {
            'baseline': bs_name,
            'problem': problem,
            'solution': state['output'],
            'log': state['log']
        }
        solutions.append(solution)
        break

  0%|          | 0/32 [04:18<?, ?it/s]


In [11]:
solutions

[{'baseline': 'single-agent',
  'problem': 'Amsterdam. Develop a city branding strategy balancing tourism and local identity.',
  'solution': 'This is a critical challenge for Amsterdam. The city is a victim of its own success, facing a classic "tragedy of the commons" where mass tourism degrades the very asset (the authentic, historic, liveable city) that visitors come to see. A successful branding strategy must move from **"promotion"** to **"regulation & curation"** , and from **"visitor volume"** to **"visitor value & resident well-being."**\n\nHere is a comprehensive city branding strategy for Amsterdam, designed to balance tourism and local identity.\n\n### Core Branding Concept: "Amsterdam: Live It, Don\'t Just Visit It."\n\nThis tagline reframes the city not as a theme park to be consumed, but as a living organism to be respected and experienced on its own terms. It prioritizes depth over breadth.\n\n---\n\n### The Three Pillars of the Strategy\n\n#### Pillar 1: The "Informed L

In [15]:
for solution in solutions:
    solution['log'] = [m.model_dump() for m in solution['log']]

In [16]:
write_json(solutions, './data/solutions.json')

In [23]:
import pickle

# Open the file in binary read mode
with open('graph_drive_spb.pkl', 'rb') as file:
    # Load the object from the file
    graph = pickle.load(file)

In [24]:
data = []

for u,v,d in graph.edges(data=True):
    data.append(d)

In [27]:
data[0]

{'length_meter': 25.115,
 'time_min': 0.025,
 'geometry': <LINESTRING (350708.698 6642736.646, 350692.807 6642756.095)>,
 'lanes': '2'}

In [26]:
import geopandas as gpd

gpd.GeoDataFrame(data, crs=32636).to_file('roads.geojson')